# TB Portals â€” 04 Â· Eval / GATE verification

Aggregates `results.csv` over seeds and compares against Kantipudi A2. **Gate passes** when our reproduction lands within roughly Â±3 Timika-MAE and Â±0.1 Pearson of their numbers â€” that validates the data pipeline and you can move to Day 2 (MoE + DANN).

In [ ]:
# ── Pull latest codebase from GitHub ─────────────────────────────────────
import os, subprocess, sys

REPO_URL = "https://github.com/mabdullahi7780/dl-project-codebase.git"
REPO_DIR = "/kaggle/working/dl-project-codebase"

if os.path.isdir(REPO_DIR):
subprocess.run(["git","-C",REPO_DIR,"pull","--ff-only"],check=True)
else:
subprocess.run(["git","clone","--depth","1","--branch","cleaned-repo","https://github.com/mabdullahi7780/dl-project-codebase.git",REPO_DIR],check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("repo ready at", REPO_DIR)


In [ ]:
# --- Clone / update the repo (requires Internet enabled in Kaggle) ---
import os, subprocess, sys

REPO_URL = "https://github.com/mabdullahi7780/dl-project-codebase.git"
REPO_DIR = "/kaggle/working/dl-project-codebase"

if os.path.exists(REPO_DIR):
subprocess.run(["git","-C",REPO_DIR,"pull","--ff-only"],check=True)
else:
subprocess.run(["git","clone","--depth","1","--branch","cleaned-repo","https://github.com/mabdullahi7780/dl-project-codebase.git",REPO_DIR],check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("repo ready at", REPO_DIR)

In [ ]:
import sys, os
REPO_DIR = '/kaggle/working/dl-project-codebase'
# Ensure repo is cloned and on sys.path
if not os.path.isdir(REPO_DIR):
    import subprocess
subprocess.run(["git","clone","--depth","1","--branch","cleaned-repo","https://github.com/mabdullahi7780/dl-project-codebase.git",REPO_DIR],check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import sys
import pandas as pd, numpy as np
REPO_DIR = '/kaggle/working/dl-project-codebase'
WORK = '/kaggle/working'
OUT_DIR = f'{WORK}/checkpoints/tbportals/baseline'
sys.path.insert(0, REPO_DIR)
from src.evaluation.eval_tbportals import KANTIPUDI_A2

res = pd.read_csv(f'{OUT_DIR}/results.csv')
res.head()

In [ ]:
rows = []
for country in ['Romania', 'Moldova', 'Kazakhstan']:
    sub = res[res.held_out == country]
    if len(sub) == 0:
        continue
    k = KANTIPUDI_A2[country]
    rows.append({
        'country': country, 'n_runs': len(sub),
        'timika_mae': f"{sub.timika_mae.mean():.2f}+/-{sub.timika_mae.std(ddof=0):.2f}",
        'timika_mae_K': k['timika_mae'],
        'timika_pearson': f"{sub.timika_pearson.mean():.2f}",
        'pearson_K': k['timika_pearson'],
        'alp_mae': f"{sub.alp_mae.mean():.2f}", 'alp_mae_K': k['alp_mae'],
        'cavity_auc': f"{sub.cavity_auc.mean():.2f}", 'cavity_auc_K': k['cavity_auc'],
    })
pd.DataFrame(rows)

In [ ]:
# Programmatic gate verdict
ok = True
for country in ['Romania', 'Moldova', 'Kazakhstan']:
    sub = res[res.held_out == country]
    if len(sub) == 0:
        print(f'{country}: NO RUNS'); ok = False; continue
    k = KANTIPUDI_A2[country]
    dmae = sub.timika_mae.mean() - k['timika_mae']
    dpear = sub.timika_pearson.mean() - k['timika_pearson']
    verdict = 'PASS' if (dmae <= 3.0 and dpear >= -0.1) else 'REVIEW'
    if verdict != 'PASS':
        ok = False
    print(f'{country}: Timika MAE delta={dmae:+.2f}, Pearson delta={dpear:+.2f} -> {verdict}')
print('\nDAY-1 GATE:', 'PASSED â€” proceed to Day 2' if ok else 'REVIEW â€” check ALP scale / country join / patient leakage')